# Football Brothers – Player Similarity Model

This notebook builds a simple player similarity model using Kaggle football datasets (21/22 and 22/23 seasons).


Links:
https://www.kaggle.com/datasets/vivovinco/20212022-football-player-stats/data,
https://www.kaggle.com/datasets/vivovinco/20222023-football-player-stats/data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

sys.path.append(str(Path("..").resolve()))

from src.data_prepare import prepare_base_df, add_per90

DATA_DIR = Path("..") / "data"
MIN_MINUTES = 900

df_base = prepare_base_df(DATA_DIR, min_minutes=MIN_MINUTES, drop_gk=True)

df_base.shape, df_base.columns[:]

((2511, 123),
 Index(['2CrdY', '90s', 'AerLost', 'AerWon', 'AerWon%', 'Age', 'Assists',
        'BlkPass', 'BlkSh', 'Blocks',
        ...
        'TouAtt3rd', 'TouAttPen', 'TouDef3rd', 'TouDefPen', 'TouLive',
        'TouMid3rd', 'Touches', 'Pos_tags', 'Is_hybrid', 'Pos_group'],
       dtype='str', length=123))

## Feature Grouping Strategy

Players are compared using grouped football metrics.
Each group has a custom weight.

In [2]:
FEATURE_GROUPS = {
    # 1) shooting threat
    "threat": {
        "weight": 0.18,
        "per90": ["Goals", "Shots", "SoT"],
        "raw": ["SoT%", "G/Sh", "ShoDist"],
    },

    # 2) Creation
    "creation": {
        "weight": 0.15,
        "per90": ["Assists", "PasAss", "SCA", "GCA", "PPA", "CrsPA"],
        "raw": [],
    },

    # 3A) passing progression
    "progression_pass": {
        "weight": 0.10,
        "per90": ["PasProg", "Pas3rd", "TB", "Sw", "PasTotPrgDist"],
        "raw": [],
    },

    # 3B) carry progression
    "progression_carry": {
        "weight": 0.08,
        "per90": ["CarProg", "CarPrgDist", "CPA", "RecProg"],
        "raw": [],
    },

    # 4A) passing volume
    "passing_volume": {
        "weight": 0.09,
        "per90": ["PasTotAtt", "PasTotCmp", "PasTotDist"],
        "raw": [],
    },

    # 4B) quality of passes
    "passing_quality": {
        "weight": 0.05,
        "per90": [],
        "raw": ["PasTotCmp%"],
    },

    # 5) Touch profiles - role
    "touch_profile": {
        "weight": 0.10,
        "per90": ["Touches", "TouDef3rd", "TouMid3rd", "TouAtt3rd", "TouAttPen"],
        "raw": [],
    },

    # 6) Defensywa
    "defense_actions": {
        "weight": 0.15,
        "per90": ["Tkl", "Int", "Tkl+Int", "Blocks", "Clr", "Err"],
        "raw": [],
    },

    # 7) Aerial duels
    "aerial": {
        "weight": 0.06,
        "per90": ["AerWon", "AerLost"],
        "raw": ["AerWon%"],
    },

    # 8) Ball security
    "ball_security": {
        "weight": 0.04,
        "per90": ["CarMis", "CarDis"],
        "raw": [],
    },
}

In [3]:
per90_cols = sorted({c for g in FEATURE_GROUPS.values() for c in g["per90"]})
raw_cols  = sorted({c for g in FEATURE_GROUPS.values() for c in g["raw"]})

missing = [c for c in (per90_cols + raw_cols) if c not in df_base.columns]
print("Missing features:", missing)

per90_cols = [c for c in per90_cols if c in df_base.columns]
raw_cols = [c for c in raw_cols if c in df_base.columns]

print("Using per90:", len(per90_cols), "Using raw:", len(raw_cols))

Missing features: []
Using per90: 36 Using raw: 5


## Feature Matrix Construction

Selected features are transformed to per90 metrics where needed.
Final matrix is built using weighted feature groups.

In [4]:
for c in raw_cols:
    df_base[c] = pd.to_numeric(df_base[c], errors="coerce")

df_base = add_per90(df_base, per90_cols)

feature_cols_final = [c + "_per90" for c in per90_cols] + raw_cols
X = df_base[feature_cols_final].copy()

print("X shape:", X.shape)
X.isna().mean().sort_values(ascending=False).head(10)

X shape: (2511, 41)


AerLost_per90       0.0
AerWon_per90        0.0
Assists_per90       0.0
Blocks_per90        0.0
CPA_per90           0.0
CarDis_per90        0.0
CarMis_per90        0.0
CarPrgDist_per90    0.0
CarProg_per90       0.0
Clr_per90           0.0
dtype: float64

In [5]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

imputer = SimpleImputer(strategy="median")
scaler = RobustScaler()

X_imp = imputer.fit_transform(X)
X_scaled = scaler.fit_transform(X_imp)

X_scaled.shape

(2511, 41)

In [6]:
import numpy as np

X_weighted = X_scaled.copy()
col_index = {col: i for i, col in enumerate(feature_cols_final)}

for group_name, cfg in FEATURE_GROUPS.items():
    w = cfg["weight"]
    cols = [c + "_per90" for c in cfg["per90"]] + cfg["raw"]
    cols = [c for c in cols if c in feature_cols_final]
    factor = np.sqrt(w)
    for c in cols:
        X_weighted[:, col_index[c]] *= factor

X_weighted.shape

(2511, 41)

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(X_weighted)
sim.shape

(2511, 2511)

## Similarity Modeling

Cosine similarity is used to measure stylistic similarity between players.
It focuses on the direction of a player’s statistical profile rather than the absolute magnitude of the values.

In [8]:
def find_similar(player_name: str, season: str, top_n: int = 100, power: int = 3):
    mask = (df_base["Player"] == player_name) & (df_base["Season"] == season)
    if mask.sum() == 0:
        raise ValueError("Nie znaleziono takiego Player + Season.")

    idx = df_base.index[mask][0]
    idx_pos = df_base.index.get_loc(idx)

    scores = sim[idx_pos].copy()

    # don't show yourself
    scores[idx_pos] = -1

    # don't show different season of same player
    same_player = (df_base["Player"].values == player_name)
    scores[same_player] = -1

    # calibration for better difference in similarity
    scores_cal = scores.copy()
    pos = scores_cal > 0
    scores_cal[pos] = scores_cal[pos] ** power

    top_idx = np.argsort(scores_cal)[::-1][:top_n]

    result = df_base.iloc[top_idx][
        ["Player", "Season", "Squad", "Comp", "Pos", "Min"]
    ].copy()

    result["similarity"] = scores_cal[top_idx].round(3)

    return result.sort_values("similarity", ascending=False)

In [9]:
df_base[["Player","Season"]].head(100)

,Player,Season
0,Max Aarons,2021/22
1,Yunis Abdelhamid,2021/22
2,Salis Abdul Samed,2021/22
3,Laurent Abergel,2021/22
4,Tammy Abraham,2021/22
...,...,...
95,Stéphane Bahoken,2021/22
96,Nedim Bajrami,2021/22
97,Mitchel Bakker,2021/22
98,Ridle Baku,2021/22


In [11]:
find_similar("Nico Williams", "2021/22", top_n=10)

,Player,Season,Squad,Comp,Pos,Min,similarity
1389,Adama Traoré,2021/22,Wolves,Premier League,FW,1067,0.721
769,Takefusa Kubo,2021/22,Mallorca,La Liga,FWMF,1605,0.687
547,Jack Grealish,2021/22,Manchester City,Premier League,FW,1914,0.616
1131,Marko Pjaca,2021/22,Torino,Serie A,MFFW,1149,0.591
1586,Lameck Banda,2022/23,Lecce,Serie A,FWMF,1051,0.576
41,Jim Allevinah,2021/22,Clermont Foot,Ligue 1,FW,1488,0.570
639,Callum Hudson-Odoi,2021/22,Chelsea,Premier League,FWMF,962,0.570
653,Jonathan Ikone,2021/22,Lille,Ligue 1,MFFW,1062,0.566
1311,Riccardo Sottil,2021/22,Fiorentina,Serie A,FW,1239,0.553
1677,Yannick Carrasco,2022/23,Atlético Madrid,La Liga,DFMF,969,0.542


### Example: Nico Williams (2021/22)

The model returns mostly attacking wingers and wide forwards such as Adama Traoré, Takefusa Kubo or Jack Grealish.
These players share a similar profile based on ball progression, carries and attacking involvement in wide areas.

The results look reasonable, as Nico Williams is primarily a direct winger focused on dribbling, progression and attacking contribution from the flanks.

In [12]:
def compare_players(base_player: str, base_season: str, top_n: int = 5):
    mask = (df_base["Player"] == base_player) & (df_base["Season"] == base_season)
    if mask.sum() == 0:
        raise ValueError("Nie znaleziono takiego Player + Season.")

    base_idx = df_base.index[mask][0]
    base_pos = df_base.index.get_loc(base_idx)

    scores = sim[base_pos].copy()

    scores[base_pos] = -1
    same_player = (df_base["Player"].values == base_player)
    scores[same_player] = -1

    top_idx_pos = np.argsort(scores)[::-1][:top_n]
    top_indices = df_base.index[top_idx_pos]


    selected_rows = df_base.loc[[base_idx] + list(top_indices)]
    stats_table = selected_rows[["Player", "Season"] + feature_cols_final].copy()

    return stats_table.round(3)

In [13]:
compare_players("Nico Williams", "2021/22", top_n=5)

,Player,Season,AerLost_per90,AerWon_per90,Assists_per90,Blocks_per90,CPA_per90,CarDis_per90,CarMis_per90,CarPrgDist_per90,...,TouAtt3rd_per90,TouAttPen_per90,TouDef3rd_per90,TouMid3rd_per90,Touches_per90,AerWon%,G/Sh,PasTotCmp%,ShoDist,SoT%
1470,Nico Williams,2021/22,0.068,0.050,0.000,0.108,0.140,0.144,0.171,10.007,...,1.872,0.397,0.374,1.148,3.154,42.3,0.00,71.2,15.5,30.3
1389,Adama Traoré,2021/22,0.183,0.085,0.000,0.056,0.205,0.247,0.261,16.076,...,1.966,0.396,0.403,1.639,3.513,31.6,0.04,74.5,18.9,26.1
769,Takefusa Kubo,2021/22,0.057,0.041,0.000,0.101,0.072,0.133,0.133,7.567,...,1.270,0.234,0.363,1.107,2.511,41.9,0.02,70.3,19.8,18.4
547,Jack Grealish,2021/22,0.066,0.035,0.007,0.044,0.194,0.095,0.057,10.310,...,1.746,0.428,0.163,1.014,2.681,34.8,0.07,86.5,14.3,35.6
1131,Marko Pjaca,2021/22,0.226,0.122,0.000,0.080,0.116,0.213,0.189,8.484,...,1.914,0.342,0.262,1.648,3.547,35.1,0.14,77.2,16.3,42.9
1586,Lameck Banda,2022/23,0.117,0.037,0.008,0.132,0.168,0.161,0.278,10.145,...,1.769,0.329,0.373,1.239,3.282,23.8,0.05,62.6,19.5,33.3


## Some examples


In [15]:
find_similar("Erling Haaland", "2022/23", top_n=8)

,Player,Season,Squad,Comp,Pos,Min,similarity
1979,Harry Kane,2022/23,Tottenham,Premier League,FW,2055,0.976
2206,Victor Osimhen,2022/23,Napoli,Serie A,FW,1401,0.976
2152,Terem Moffi,2022/23,Lorient,Ligue 1,FW,1386,0.969
1584,Folarin Balogun,2022/23,Reims,Ligue 1,FW,1591,0.968
2019,Alexandre Lacazette,2022/23,Lyon,Ligue 1,FW,1923,0.968
1804,Breel Embolo,2022/23,Monaco,Ligue 1,FW,1542,0.960
2427,Ivan Toney,2022/23,Brentford,Premier League,FW,1789,0.959
1743,Jonathan David,2022/23,Lille,Ligue 1,FW,1759,0.953


### Erling Haaland (2022/23)

The model returns mostly classic strikers such as Harry Kane, Victor Osimhen or Ivan Toney.
This matches Haaland’s profile as a goal-focused centre-forward with strong scoring and finishing statistics.

In [16]:
find_similar("Memphis Depay", "2021/22", top_n=8)

,Player,Season,Squad,Comp,Pos,Min,similarity
952,Dries Mertens,2021/22,Napoli,Serie A,FWMF,1384,0.740
467,Phil Foden,2021/22,Manchester City,Premier League,FW,2128,0.700
549,Mason Greenwood,2021/22,Manchester Utd,Premier League,FWMF,1271,0.693
1407,Cengiz Ünder,2021/22,Marseille,Ligue 1,FWMF,2200,0.658
555,Arnaut Groeneveld,2021/22,Villarreal,La Liga,FW,1472,0.652
116,Musa Barrow,2021/22,Bologna,Serie A,FWMF,2134,0.649
869,Donyell Malen,2021/22,Dortmund,Bundesliga,FW,1684,0.633
861,Riyad Mahrez,2021/22,Manchester City,Premier League,FW,1498,0.628


### Example: Memphis Depay (2021/22)

The model returns mainly hybrid forwards and technical attacking players such as Dries Mertens, Phil Foden or Riyad Mahre.
This makes sense, as Depay’s profile is not that of a classic striker, but rather a mobile forward combining scoring, ball progression and creative involvement.

In [17]:
find_similar("João Félix", "2021/22", top_n=8)

,Player,Season,Squad,Comp,Pos,Min,similarity
987,Gerard Moreno,2021/22,Villarreal,La Liga,FW,1191,0.787
1003,Luis Muriel,2021/22,Atalanta,Serie A,FW,1539,0.670
1476,Florian Wirtz,2021/22,Leverkusen,Bundesliga,MFFW,1851,0.666
1029,Neymar,2021/22,Paris S-G,Ligue 1,FWMF,1851,0.666
1335,Isaac Success,2021/22,Udinese,Serie A,FW,911,0.662
652,Kelechi Iheanacho,2021/22,Leicester City,Premier League,FW,1264,0.643
1244,Oihan Sancet,2021/22,Athletic Club,La Liga,FW,1422,0.631
363,Ángel Di María,2021/22,Paris S-G,Ligue 1,FW,1645,0.628


### Example: João Félix (2021/22)

João Félix's profile is defined more by link-up play, creativity and high dribbling involvement rather than pure striker or winger output. Because of his hybrid attacking role, his profile is harder to categorize, which is reflected in the results where we can see a mix of attacking midfielders, creative strikers and wingers.

In [18]:
find_similar("Kevin De Bruyne", "2021/22", top_n=8)

,Player,Season,Squad,Comp,Pos,Min,similarity
870,Ruslan Malinovskyi,2021/22,Atalanta,Serie A,FWMF,1592,0.747
363,Ángel Di María,2021/22,Paris S-G,Ligue 1,FW,1645,0.742
1029,Neymar,2021/22,Paris S-G,Ligue 1,FWMF,1851,0.707
987,Gerard Moreno,2021/22,Villarreal,La Liga,FW,1191,0.665
577,Ýlkay Gündoðan,2021/22,Manchester City,Premier League,MF,1857,0.641
1135,Daniel Podence,2021/22,Wolves,Premier League,FWMF,1484,0.632
657,Lorenzo Insigne,2021/22,Napoli,Serie A,FW,2290,0.624
629,Jonas Hofmann,2021/22,M'Gladbach,Bundesliga,MFFW,2078,0.624


### Kevin De Bruyne (2021/22)

The model returns mostly creative attacking players such as Ruslan Malinovskyi, Ángel Di María or Neymar, which fits De Bruyne's profile as a highly creative attacking midfielder.

Interestingly, Gerard Moreno (in season 2021/22 – 24 games as striker) appears again among similar players. Even though he played mainly as a striker, his statistical profile includes a lot of chance creation and creative involvement, which makes him comparable to more creative roles.

In [19]:
find_similar("Rodri", "2022/23", top_n=8)

,Player,Season,Squad,Comp,Pos,Min,similarity
2504,Oleksandr Zinchenko,2022/23,Arsenal,Premier League,DF,1081,0.666
1505,Oleksandr Zinchenko,2021/22,Manchester City,Premier League,DF,1047,0.614
2309,Fabián Ruiz Peña,2022/23,Paris S-G,Ligue 1,MF,941,0.603
33,Thiago Alcántara,2021/22,Liverpool,Premier League,MF,1534,0.593
2219,Thomas Partey,2022/23,Arsenal,Premier League,MF,1539,0.583
1551,Sofyan Amrabat,2022/23,Fiorentina,Serie A,MF,1165,0.572
457,Fernandinho,2021/22,Manchester City,Premier League,MFDF,964,0.558
1852,Johan Gastien,2022/23,Clermont Foot,Ligue 1,MF,1564,0.533


### Rodri (2022/23)

For Rodri, we would expect players with a deep midfield profile focused on high passing volume, passing efficiency and defensive contribution.

The results reflect this quite well, with players such as Fabián Ruiz, Thiago Alcântara, Thomas Partey or Sofyan Amrabat appearing among the most similar profiles. These players share a role centered around controlling possession, progressing the ball and maintaining midfield stability.

It is also interesting that Oleksandr Zinchenko appears twice (in two different seasons). Although he is often listed as a defender, his inverted fullback role involves many midfield-like responsibilities, which makes his statistical profile closer to deep midfielders like Rodri than to classic or attacking fullbacks. It is also worth mentioning that his appearances come from two different clubs, which suggests that his role remained very similar across both teams between the 2021 and 2023 seasons.

In [20]:
find_similar("Nampalys Mendy", "2021/22", top_n=8)

,Player,Season,Squad,Comp,Pos,Min,similarity
1846,Idrissa Gana Gueye,2022/23,Everton,Premier League,MF,1221,0.800
2210,Salih Özcan,2022/23,Dortmund,Bundesliga,MF,1180,0.642
1030,Yvan Neyou,2021/22,Saint-Étienne,Ligue 1,MF,936,0.634
560,Nemanja Gudelj,2021/22,Sevilla,La Liga,MFDF,1046,0.632
1707,Carlos Clerc,2022/23,Elche,La Liga,DFMF,1172,0.631
50,Sofyan Amrabat,2021/22,Fiorentina,Serie A,MF,944,0.631
1887,Ilia Gruev,2022/23,Werder Bremen,Bundesliga,MF,960,0.613
1241,Albert Sambi Lokonga,2021/22,Arsenal,Premier League,MF,1134,0.580


### Nampalys Mendy (2021/22)

Here the expected profile is a more classic defensive midfielder, focused on ball-winning, screening the defense and simple circulation rather than creativity or attacking involvement.

The results fit this idea quite well, with players such as Idrissa Gueye, Salih Özcan, Nemanja Gudelj or Sofyan Amrabat appearing among the most similar profiles. Compared to Rodri, this group looks less like deep playmakers and more like traditional defensive midfielders whose main role is to provide stability and defensive coverage.

An interesting observation is that Sofyan Amrabat appears again in the Rodri example, but from a different season (2022/23).
This may indicate a potential shift in his role between seasons. In the Rodri comparison Amrabat appears among more possession-oriented midfielders, while in the Nampalys Mendy example he appears among more traditional defensive midfielders.
This raises the question whether his role in 2022/23 involved a greater participation in build-up play and ball progression compared to 2021/22. To explore this, the next step is to compare the statistical profiles of both seasons.

In [25]:
def compare_two_player_seasons(player_1, season_1, player_2, season_2):
    mask_1 = (df_base["Player"] == player_1) & (df_base["Season"] == season_1)
    mask_2 = (df_base["Player"] == player_2) & (df_base["Season"] == season_2)

    if mask_1.sum() == 0:
        raise ValueError(f"Nie znaleziono: {player_1} ({season_1})")
    if mask_2.sum() == 0:
        raise ValueError(f"Nie znaleziono: {player_2} ({season_2})")

    idx_1 = df_base.index[mask_1][0]
    idx_2 = df_base.index[mask_2][0]

    row_1 = df_base.loc[[idx_1]]
    row_2 = df_base.loc[[idx_2]]

    comparison = pd.concat([row_1, row_2])[["Player", "Season"] + feature_cols_final].copy()
    return comparison.round(3)

In [26]:
compare_two_player_seasons("Sofyan Amrabat", "2021/22", "Sofyan Amrabat", "2022/23")

,Player,Season,AerLost_per90,AerWon_per90,Assists_per90,Blocks_per90,CPA_per90,CarDis_per90,CarMis_per90,CarPrgDist_per90,...,TouAtt3rd_per90,TouAttPen_per90,TouDef3rd_per90,TouMid3rd_per90,Touches_per90,AerWon%,G/Sh,PasTotCmp%,ShoDist,SoT%
50,Sofyan Amrabat,2021/22,0.154,0.090,0.0,0.245,0.01,0.064,0.046,10.610,...,1.105,0.018,1.714,5.038,7.295,37.0,0.25,90.3,24.4,75.0
1551,Sofyan Amrabat,2022/23,0.042,0.042,0.0,0.084,0.00,0.078,0.102,11.465,...,1.039,0.018,0.915,4.116,5.984,50.0,0.00,88.8,26.3,28.6


The data partially supports the hypothesis. In the 2022/23 season Amrabat produced significantly more progressive passes (0.721 vs 0.435 per 90), suggesting greater involvement in build-up play.
However, in 2021/22 he recorded more defensive actions (0.371 vs 0.222 tackles + interceptions per 90) and more progressive carries. This indicates that while his role in 2021/22 was more defensively oriented, progression often came through ball carrying rather than passing.

In [21]:
find_similar("Conor Coady", "2021/22", top_n=8)

,Player,Season,Squad,Comp,Pos,Min,similarity
445,Wout Faes,2021/22,Reims,Ligue 1,DF,3330,0.745
1300,Milan kriniar,2021/22,Inter,Serie A,DF,3150,0.707
1392,Ismaël Traoré,2021/22,Angers,Ligue 1,DF,3215,0.695
969,Stefan Mitrovi?,2021/22,Getafe,La Liga,DF,2667,0.676
812,Philipp Lienhart,2021/22,Freiburg,Bundesliga,DF,2870,0.632
1219,Amir Rrahmani,2021/22,Napoli,Serie A,DF,2927,0.628
563,Marc Guéhi,2021/22,Crystal Palace,Premier League,DF,3222,0.622
780,Víctor Laguardia,2021/22,Alavés,La Liga,DF,3042,0.561


In [22]:
find_similar("Rúben Dias", "2021/22", top_n=8)

,Player,Season,Squad,Comp,Pos,Min,similarity
788,Aymeric Laporte,2021/22,Manchester City,Premier League,DF,2828,0.856
915,Joël Matip,2021/22,Liverpool,Premier League,DF,2790,0.706
730,Presnel Kimpembe,2021/22,Paris S-G,Ligue 1,DF,2576,0.603
934,Facundo Medina,2021/22,Lens,Ligue 1,DF,2593,0.587
1130,Gerard Piqué,2021/22,Barcelona,La Liga,DF,2092,0.559
1234,William Saliba,2021/22,Marseille,Ligue 1,DF,3240,0.492
619,Lucas Hernández,2021/22,Bayern Munich,Bundesliga,DF,2030,0.491
1290,Thiago Silva,2021/22,Chelsea,Premier League,DF,2650,0.460


### Centre-back profiles: Conor Coady vs Rúben Dias

Both examples represent centre-backs, but the groups of similar players suggest two slightly different defensive profiles.

For Conor Coady the most similar players tend to come from teams that are not dominant in possession. Clubs such as Reims, Angers, Getafe or Alavés (as well as two Serie A teams, a league often associated with a more defensive and structured style of play) are generally more defensive and spend more time without the ball. As a result, their centre-backs are often more focused on defensive actions such as positioning, clearances and blocking rather than on building attacks from the back.

In contrast, the players appearing in the Rúben Dias example mostly come from bigger clubs that naturally aim to dominate possession and control the game. In these teams centre-backs are expected not only to defend but also to actively participate in build-up play and ball progression.

This difference highlights how even a simple model can distinguish between two common centre-back archetypes: more traditional defensive centre-backs and ball-playing centre-backs who are heavily involved in possession and progression.

In [23]:
find_similar("Federico Valverde", "2022/23", top_n=8)

,Player,Season,Squad,Comp,Pos,Min,similarity
1884,Vincenzo Grifo,2022/23,Freiburg,Bundesliga,FWMF,1289,0.898
2077,James Maddison,2022/23,Leicester City,Premier League,MFFW,1273,0.838
2010,Andrej Kramari?,2022/23,Hoffenheim,Bundesliga,MFFW,1365,0.833
2185,Christopher Nkunku,2022/23,RB Leipzig,Bundesliga,FWMF,1282,0.823
2059,Ademola Lookman,2022/23,Atalanta,Serie A,FWMF,1299,0.819
1591,Nicolò Barella,2022/23,Inter,Serie A,MF,1609,0.810
1614,Karim Benzema,2022/23,Real Madrid,La Liga,FW,1046,0.806
2045,Robert Lewandowski,2022/23,Barcelona,La Liga,FW,1334,0.802


### Explaining similarity features for Federico Valverde

To better understand why the model selected these players as most similar to Federico Valverde, the returned comparison table is analysed feature by feature. The selected statistics are scaled with RobustScaler, and then the absolute difference between Valverde and each similar player is computed.

The goal of this step is to identify the five features with the smallest differences for every player. These are the statistical areas in which each player is closest to Valverde, and therefore the most likely explanation for why the cosine similarity model linked them together.

In [30]:
compare_players("Federico Valverde", "2022/23", top_n=8)

,Player,Season,AerLost_per90,AerWon_per90,Assists_per90,Blocks_per90,CPA_per90,CarDis_per90,CarMis_per90,CarPrgDist_per90,...,TouAtt3rd_per90,TouAttPen_per90,TouDef3rd_per90,TouMid3rd_per90,Touches_per90,AerWon%,G/Sh,PasTotCmp%,ShoDist,SoT%
2450,Federico Valverde,2022/23,0.017,0.059,0.007,0.055,0.014,0.021,0.048,7.265,...,1.676,0.114,0.419,1.735,3.800,77.3,0.15,87.9,24.1,36.6
1884,Vincenzo Grifo,2022/23,0.039,0.029,0.015,0.015,0.049,0.064,0.098,7.035,...,1.748,0.166,0.489,2.049,4.238,42.9,0.16,74.1,22.9,41.9
2077,James Maddison,2022/23,0.035,0.010,0.025,0.040,0.070,0.070,0.121,6.270,...,1.723,0.287,0.392,1.894,3.986,22.2,0.20,77.0,20.2,33.3
2010,Andrej Kramari?,2022/23,0.013,0.026,0.013,0.035,0.026,0.091,0.139,3.829,...,1.118,0.260,0.277,1.711,3.079,66.7,0.15,78.7,15.8,39.4
2185,Christopher Nkunku,2022/23,0.060,0.060,0.000,0.030,0.070,0.124,0.149,3.915,...,1.444,0.372,0.238,1.282,2.930,50.0,0.24,80.6,15.8,41.5
2059,Ademola Lookman,2022/23,0.044,0.015,0.019,0.062,0.121,0.188,0.169,6.479,...,1.625,0.434,0.212,1.292,3.090,25.0,0.24,76.1,16.0,39.5
1591,Nicolò Barella,2022/23,0.022,0.044,0.016,0.056,0.022,0.078,0.094,6.324,...,1.173,0.125,0.564,1.754,3.436,66.7,0.29,80.7,19.8,47.1
1614,Karim Benzema,2022/23,0.037,0.037,0.022,0.059,0.111,0.104,0.127,7.069,...,2.957,0.647,0.238,1.776,4.940,50.0,0.13,85.4,16.2,34.0
2045,Robert Lewandowski,2022/23,0.096,0.101,0.018,0.046,0.036,0.086,0.210,2.932,...,1.527,0.452,0.073,0.932,2.514,51.2,0.22,76.5,12.8,51.6


In [49]:
df_valverde = compare_players("Federico Valverde", "2022/23", top_n=8).reset_index(drop=True)

In [50]:
from sklearn.preprocessing import RobustScaler
meta_cols = ["Player", "Season"]

feature_cols = [c for c in df_valverde.columns if c not in meta_cols]
X = df_valverde[feature_cols]

scaler = RobustScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=feature_cols
)

valverde = X_scaled.iloc[0]
rows = []

for i in range(1, len(df_valverde)):

    player = df_valverde.loc[i, "Player"]

    diff = (valverde - X_scaled.iloc[i]).abs().sort_values()

    rows.append({
        "player": player,
        "most_similar_features": ", ".join(diff.index[:5])
    })

explain_similarity = pd.DataFrame(rows)
explain_similarity

,player,most_similar_features
0,Vincenzo Grifo,"Err_per90, Pas3rd_per90, CarPrgDist_per90, G/S..."
1,James Maddison,"GCA_per90, Err_per90, CarProg_per90, PasTotAtt..."
2,Andrej Kramari?,"Err_per90, G/Sh, Shots_per90, Pas3rd_per90, To..."
3,Christopher Nkunku,"Err_per90, AerWon_per90, PPA_per90, PasAss_per..."
4,Ademola Lookman,"Err_per90, PasProg_per90, TouAtt3rd_per90, Pas..."
5,Nicolò Barella,"Err_per90, TouMid3rd_per90, TouAttPen_per90, P..."
6,Karim Benzema,"Err_per90, PasTotDist_per90, PasTotPrgDist_per..."
7,Robert Lewandowski,"Err_per90, PPA_per90, GCA_per90, PasAss_per90,..."


### Interpretation of results

The output shows that Valverde’s similarity to other players is mainly driven by progression, attacking involvement and creative contribution. Features such as progressive passing, progressive carrying, touches in advanced zones, shot creation and passing into dangerous areas appear repeatedly across the table.

This suggests that the model sees Valverde as a hybrid midfielder, combining ball progression with offensive activity. That is why he is linked not only with midfielders such as Barella or Maddison, but also with more attacking players such as Nkunku, Benzema or Lewandowski. The similarity is therefore based more on style of play than on nominal position.

In [52]:
find_similar("Marcos Llorente", "2021/22", top_n=8)

,Player,Season,Squad,Comp,Pos,Min,similarity
1501,Akim Zedadka,2021/22,Clermont Foot,Ligue 1,DF,3337,0.686
313,Marc Cucurella,2021/22,Brighton,Premier League,DF,3089,0.675
0,Max Aarons,2021/22,Norwich City,Premier League,DF,2881,0.596
102,Iván Balliu,2021/22,Rayo Vallecano,La Liga,DF,3147,0.558
540,Andoni Gorosabel,2021/22,Real Sociedad,La Liga,DF,2289,0.534
1435,Nacho Vidal,2021/22,Osasuna,La Liga,DF,2902,0.525
491,Javi Galán,2021/22,Celta Vigo,La Liga,DF,3298,0.524
1099,Pedrosa,2021/22,Espanyol,La Liga,DF,2720,0.523


In [53]:
compare_players("Marcos Llorente", "2021/22", top_n=8)

,Player,Season,AerLost_per90,AerWon_per90,Assists_per90,Blocks_per90,CPA_per90,CarDis_per90,CarMis_per90,CarPrgDist_per90,...,TouAtt3rd_per90,TouAttPen_per90,TouDef3rd_per90,TouMid3rd_per90,Touches_per90,AerWon%,G/Sh,PasTotCmp%,ShoDist,SoT%
820,Marcos Llorente,2021/22,0.039,0.034,0.003,0.057,0.019,0.043,0.032,3.423,...,0.765,0.084,0.548,0.956,2.140,46.3,0.00,80.2,20.6,12.5
1501,Akim Zedadka,2021/22,0.026,0.021,0.002,0.045,0.007,0.042,0.036,3.423,...,0.590,0.033,0.523,0.895,1.873,44.6,0.00,80.3,21.5,18.8
313,Marc Cucurella,2021/22,0.039,0.046,0.001,0.061,0.008,0.027,0.021,4.385,...,0.790,0.056,0.638,1.015,2.289,54.0,0.06,81.6,20.9,16.7
0,Max Aarons,2021/22,0.050,0.015,0.002,0.084,0.013,0.029,0.026,3.803,...,0.469,0.028,0.728,0.744,1.812,22.7,0.00,75.5,20.5,15.4
102,Iván Balliu,2021/22,0.031,0.030,0.002,0.040,0.007,0.022,0.026,2.740,...,0.563,0.029,0.463,0.829,1.743,49.3,0.07,76.4,20.3,20.0
540,Andoni Gorosabel,2021/22,0.057,0.056,0.006,0.057,0.006,0.034,0.033,4.315,...,0.768,0.051,0.689,1.335,2.630,49.3,0.00,83.5,20.1,0.0
1435,Nacho Vidal,2021/22,0.038,0.076,0.001,0.078,0.004,0.020,0.030,1.606,...,0.522,0.039,0.562,0.839,1.851,66.9,0.00,72.0,19.1,9.5
491,Javi Galán,2021/22,0.036,0.022,0.002,0.051,0.013,0.048,0.033,4.825,...,0.623,0.032,0.579,0.973,1.995,38.5,0.00,78.4,27.3,26.3
1099,Pedrosa,2021/22,0.033,0.021,0.001,0.057,0.019,0.044,0.059,4.099,...,0.560,0.063,0.639,0.808,1.848,38.8,0.04,75.5,23.2,28.0


In [ ]:
df_llorente = compare_players("Marcos Llorente", "2021/22", top_n=8).reset_index(drop=True)